In [0]:
# Databricks notebook source

# COMMAND ----------
# Widgets
dbutils.widgets.removeAll()

dbutils.widgets.text("catalog_name", "proyectoFinal")
dbutils.widgets.text("gold_schema", "gold")

# COMMAND ----------
# Leer parámetros
catalog_name = dbutils.widgets.get("catalog_name")
gold_schema = dbutils.widgets.get("gold_schema")

print("Configuración:")
print(f"catalog_name={catalog_name}")
print(f"gold_schema={gold_schema}")

# COMMAND ----------
# Usar catálogo y esquema
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {gold_schema}")

print("Catálogo y schema seleccionados")

# COMMAND ----------
# Vista financiera (Power BI)
finance_view_sql = f"""
CREATE OR REPLACE VIEW {catalog_name}.{gold_schema}.vw_powerbi_movie_finance AS
SELECT
    m.movie_id,
    m.title AS movie_title,
    d.director_name,
    l.language_code AS language_name,
    dt.full_date AS release_date,
    dt.year AS year_number,
    f.budget_usd,
    f.revenue_usd,
    f.profit_usd,
    f.roi_pct,
    f.margin_pct,
    f.runtime_total_min,
    f.user_score,
    f.vote_count
FROM {catalog_name}.{gold_schema}.fact_movie_metrics f
JOIN {catalog_name}.{gold_schema}.dim_movie m
    ON f.movie_key = m.movie_key
LEFT JOIN {catalog_name}.{gold_schema}.dim_director d
    ON m.director_key = d.director_key
LEFT JOIN {catalog_name}.{gold_schema}.dim_language l
    ON m.language_key = l.language_key
LEFT JOIN {catalog_name}.{gold_schema}.dim_date dt
    ON f.release_date_key = dt.date_key
"""

print("Creando vista vw_powerbi_movie_finance...")
spark.sql(finance_view_sql)

# COMMAND ----------
# Vista de géneros (Power BI)
genre_view_sql = f"""
CREATE OR REPLACE VIEW {catalog_name}.{gold_schema}.vw_powerbi_movie_genre AS
SELECT
    m.movie_key,
    m.movie_id,
    m.title,
    g.genre_name
FROM {catalog_name}.{gold_schema}.dim_movie m
LEFT JOIN {catalog_name}.{gold_schema}.bridge_movie_genre b
    ON m.movie_key = b.movie_key
LEFT JOIN {catalog_name}.{gold_schema}.dim_genre g
    ON b.genre_key = g.genre_key
"""

print("Creando vista vw_powerbi_movie_genre...")
spark.sql(genre_view_sql)

# COMMAND ----------
print("Vistas Gold creadas correctamente")